In [39]:
import pandas as pd

df = pd.read_csv('./TSA.csv')
df = df[['elapsed_time', 'bedroom', 'livingroom', 'kitchen', 'bathroom', 'room']]
df

,elapsed_time,bedroom,livingroom,kitchen,bathroom,room
0,50318.4,1,0,0,0,Bedroom
1,50321.6,1,0,0,0,Bedroom
2,50323.2,1,0,0,0,Bedroom
3,50324.8,1,0,0,0,Bedroom
4,50326.4,1,0,0,0,Bedroom
...,...,...,...,...,...,...
37257,522643.2,1,0,0,0,Bedroom
37258,522659.2,1,0,0,0,Bedroom
37259,522675.2,1,0,0,0,Bedroom
37260,522691.2,1,0,0,0,Bedroom


In [32]:
df[(df['bedroom'] == 0) & (df['bathroom'] == 1)]

,elapsed_time,bedroom,livingroom,kitchen,bathroom
3554,63353.6,0,0,0,1
3555,63369.6,0,0,0,1
3556,63385.6,0,0,0,1
3557,63401.6,0,0,0,1
3558,63417.6,0,0,0,1
...,...,...,...,...,...
36697,513235.2,0,0,0,1
36698,513251.2,0,0,0,1
36699,513267.2,0,0,0,1
36700,513283.2,0,0,0,1


In [33]:
df['action_group'] = (df['room'] != df['room'].shift()).cumsum()

# Calculate the elapsed time difference for each action group
action_group_elapsed_time = df.groupby('action_group').agg(
    start_elapsed=('elapsed_time', 'first'),
    end_elapsed=('elapsed_time', 'last'),
    room=('room', 'first')
)

# Add a column for the difference in elapsed time
action_group_elapsed_time['elapsed_time_diff'] = (
    action_group_elapsed_time['end_elapsed'] - action_group_elapsed_time['start_elapsed']
)

df

KeyError: 'room'

In [40]:
df['room']

0        Bedroom
1        Bedroom
2        Bedroom
3        Bedroom
4        Bedroom
          ...   
37257    Bedroom
37258    Bedroom
37259    Bedroom
37260    Bedroom
37261    Bedroom
Name: room, Length: 37262, dtype: object

In [42]:
df['elapsed_time'] = df['elapsed_time'] / 3600

stats = df.groupby(['room'])['elapsed_time'].agg(
    average='mean',
    maximum='max',
    minimum='min'
).reset_index()
stats

,room,average,maximum,minimum
0,Bathroom,69.879321,142.583111,17.598222
1,Bedroom,64.148922,145.196444,13.977333
2,Kitchen,78.891704,142.618667,17.540444
3,Living room,87.226162,140.400889,20.616000
4,Unknown,114.684644,142.623111,17.536000


In [51]:
import pandas as pd

from datetime import datetime


df = pd.read_csv('./TSA.csv')
# Add a group identifier to detect changes in actions
df['action_group'] = (df['room'] != df['room'].shift()).cumsum()

# Calculate the elapsed time difference for each action group
action_group_elapsed_time = df.groupby('action_group').agg(
    start_elapsed=('elapsed_time', 'first'),
    end_elapsed=('elapsed_time', 'last'),
    room=('room', 'first')
)

# Add a column for the difference in elapsed time
action_group_elapsed_time['elapsed_time_diff'] = (
    action_group_elapsed_time['end_elapsed'] - action_group_elapsed_time['start_elapsed']
)

action_group_elapsed_time['elapsed_time_diff']

action_group
1      12795.2
2          0.0
3        112.0
4          0.0
5       7136.0
        ...   
161     3632.0
162       96.0
163      112.0
164        0.0
165     9248.0
Name: elapsed_time_diff, Length: 165, dtype: float64

In [52]:
elapsed_time_stats = action_group_elapsed_time.groupby('room')['elapsed_time_diff'].agg(
    average='mean',
    maximum='max',
    minimum='min'
).reset_index()

elapsed_time_stats

,room,average,maximum,minimum
0,Bathroom,4499.085714,21612.8,0.0
1,Bedroom,18269.381818,51638.4,0.0
2,Kitchen,1527.673469,9145.6,0.0
3,Living room,2714.707692,12603.2,256.0
4,Unknown,510.950000,13776.0,0.0


In [53]:
elapsed_time_stats['average'] = elapsed_time_stats['average'] / 3600
elapsed_time_stats

,room,average,maximum,minimum
0,Bathroom,1.249746,21612.8,0.0
1,Bedroom,5.074828,51638.4,0.0
2,Kitchen,0.424354,9145.6,0.0
3,Living room,0.754085,12603.2,256.0
4,Unknown,0.141931,13776.0,0.0


In [44]:
import pandas as pd

from datetime import datetime


df = pd.read_csv('./TSA.csv')
# Add a group identifier to detect changes in actions
df['action_group'] = (df['room'] != df['room'].shift()).cumsum()

# Calculate the elapsed time difference for each action group
action_group_elapsed_time = df.groupby('action_group').agg(
    start_elapsed=('elapsed_time', 'first'),
    end_elapsed=('elapsed_time', 'last'),
    room=('room', 'first')
)

# Add a column for the difference in elapsed time
action_group_elapsed_time['elapsed_time_diff'] = (
    action_group_elapsed_time['end_elapsed'] - action_group_elapsed_time['start_elapsed']
)

# Convert elapsed_time_diff to hours
action_group_elapsed_time['elapsed_time_hours'] = action_group_elapsed_time['elapsed_time_diff'] / 3600

# Calculate the statistics for each room category
elapsed_time_stats = action_group_elapsed_time.groupby('room')['elapsed_time_hours'].agg(
    average='mean',
    maximum='max',
    minimum='min'
).reset_index()

# Map average elapsed time by room to a dictionary for comparison
average_elapsed_time_by_room = elapsed_time_stats.set_index('room')['average'].to_dict()

# Calculate the cumulative time passed since the start of each action group
df['elapsed_time_cumulative'] = df.groupby('action_group')['elapsed_time'].transform(lambda x: x - x.iloc[0])

# Assign the label based on whether the cumulative elapsed time exceeds the average for the room
df['exceeds_average'] = df.apply(
    lambda row: 1 if row['elapsed_time_cumulative'] / 3600 > average_elapsed_time_by_room.get(row['room'], 0) else 0,
    axis=1
)

# Save the updated data to a new file
df.to_csv('updated_data_with_labels.csv', index=False)


In [28]:
df1 = pd.read_csv('./TS.csv')
df1 = df1[['elapsed_time', 'bedroom', 'livingroom', 'kitchen', 'bathroom']]
df1
# df1[(df1['bedroom'] == 0) & (df1['bathroom'] == 1)]



,elapsed_time,bedroom,livingroom,kitchen,bathroom
0,50326.4,1,0,0,0
1,50331.2,1,0,0,0
2,50336.0,1,0,0,0
3,50340.8,1,0,0,0
4,50342.4,1,0,0,0
...,...,...,...,...,...
33651,541691.2,1,0,0,0
33652,541692.8,1,0,0,0
33653,541696.0,1,0,0,0
33654,541697.6,1,0,0,0
